In [1]:
!pip install --no-cache-dir git+https://github.com/JorgeDeLosSantos/moro.git

  Cloning https://github.com/JorgeDeLosSantos/moro.git to /tmp/pip-req-build-1qscn5pt
  Running command git clone --filter=blob:none --quiet https://github.com/JorgeDeLosSantos/moro.git /tmp/pip-req-build-1qscn5pt
  Resolved https://github.com/JorgeDeLosSantos/moro.git to commit 5c0167e1a4fb8a5a3fe6fccd8b33be455c7370e0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for moro: filename=moro-0.4.0.dev0-py3-none-any.whl size=54310 sha256=1a7ae4c72f710f375f3e2e9549874d63570f608dc5b57ea5aeec3f706379bb28
  Stored in directory: /tmp/pip-ephem-wheel-cache-bpg5gi6y/wheels/12/3a/28/f05eb4b507d1421d710b3c38f6cd46642a31583366b2647b01
Successfully built moro


# Inverse Kinematics Examples

This notebook shows position inverse kinematics with `solve_position_ik` for:

1. A planar RR manipulator.
2. An anthropomorphic RRR manipulator.

All targets are built from a known reference configuration to guarantee reachability.

In [2]:
import numpy as np
import sympy as sp
import moro as mr

from moro.abc import q1, q2, q3, l1, l2, l3, d1
from moro.inverse_kinematics import solve_position_ik


def fk_position(robot, q_values, parameters=None):
    substitutions = dict(zip(robot.qs, q_values))
    if parameters is not None:
        substitutions.update(parameters)
    p = robot.T[:3, 3].subs(substitutions)
    return np.asarray(p, dtype=float).reshape(3)


print("=== Example 1: Planar RR manipulator ===")

RR = mr.Robot((l1, 0, 0, q1, "r"), (l2, 0, 0, q2, "r"))
rr_params = {l1: 200.0, l2: 200.0}

# Build a reachable target from a known configuration.
q_ref_rr = [np.deg2rad(30.0), np.deg2rad(60.0)]
target_rr = fk_position(RR, q_ref_rr, parameters=rr_params)

sol_rr = solve_position_ik(
    RR,
    target_rr,
    q0=[0.3, 0.4],
    parameters=rr_params,
    method="newton",
    tol=1e-9,
)

p_rr = fk_position(RR, sol_rr.q, parameters=rr_params)
err_rr = np.linalg.norm(target_rr - p_rr)

print("Target (RR):", target_rr)
print("Solution (RR):", sol_rr)
print("Message (RR):", sol_rr.message)
print("Residual (RR):", sol_rr.residual)
print("FK at solution (RR):", p_rr)
print("Validation error norm (RR):", err_rr)

print("\nReproducible random initialization (same seed):")
sol_rr_seed1 = solve_position_ik(RR, target_rr, parameters=rr_params, random_state=42)
sol_rr_seed2 = solve_position_ik(RR, target_rr, parameters=rr_params, random_state=42)
print("Seeded run #1 q:", sol_rr_seed1.q)
print("Seeded run #2 q:", sol_rr_seed2.q)
print("Same q:", np.allclose(sol_rr_seed1.q, sol_rr_seed2.q))

=== Example 1: Planar RR manipulator ===
Target (RR): [173.20508076 300.           0.        ]
Solution (RR): IKSolution(q=[0.5235987755982989, 1.0471975511965976], Converged, method=newton, iters=7, error=2.84e-14)
Message (RR): Converged successfully.
Residual (RR): [2.842170943040401e-14, 0.0, 0.0]
FK at solution (RR): [173.20508076 300.           0.        ]
Validation error norm (RR): 2.842170943040401e-14

Reproducible random initialization (same seed):
Seeded run #1 q: [1.5707963267971492, -1.0471975512007992]
Seeded run #2 q: [1.5707963267971492, -1.0471975512007992]
Same q: True


In [3]:
print("=== Example 2: Anthropomorphic RRR manipulator ===")

RRR = mr.Robot(
    (0, sp.pi / 2, d1, q1, "r"),
    (l2, 0, 0, q2, "r"),
    (l3, 0, 0, q3, "r"),
)

rrr_params = {d1: 120.0, l2: 140.0, l3: 110.0}

# Build a reachable target from a known configuration.
q_ref_rrr = [np.deg2rad(30.0), np.deg2rad(-20.0), np.deg2rad(35.0)]
target_rrr = fk_position(RRR, q_ref_rrr, parameters=rrr_params)

sol_rrr = solve_position_ik(
    RRR,
    target_rrr,
    q0=[0.3, -0.1, 0.2],
    parameters=rrr_params,
    method="lm",
    tol=1e-9,
)

p_rrr = fk_position(RRR, sol_rrr.q, parameters=rrr_params)
err_rrr = np.linalg.norm(target_rrr - p_rrr)

print("Target (RRR):", target_rrr)
print("Solution (RRR):", sol_rrr)
print("Message (RRR):", sol_rrr.message)
print("Residual (RRR):", sol_rrr.residual)
print("FK at solution (RRR):", p_rrr)
print("Validation error norm (RRR):", err_rrr)

print("\nMethod comparison on the same target:")
for method in ("lm", "newton", "ccd"):
    sol_method = solve_position_ik(
        RRR,
        target_rrr,
        q0=[0.3, -0.1, 0.2],
        parameters=rrr_params,
        method=method,
        tol=1e-8,
        max_iter=600 if method == "ccd" else 200,
    )
    p_method = fk_position(RRR, sol_method.q, parameters=rrr_params)
    err_method = np.linalg.norm(target_rrr - p_method)
    print(f"- {method:6s} converged={sol_method.converged} iter={sol_method.iterations} error={err_method:.3e}")

print("\nNote: solve_position_ik handles position only (x, y, z), not full orientation IK.")

=== Example 2: Anthropomorphic RRR manipulator ===
Target (RRR): [205.9484688 118.9044039 100.5872749]
Solution (RRR): IKSolution(q=[0.5235987755982988, -0.34906585039974874, 0.6108652382000069], Converged, method=lm, iters=7, error=7.39e-11)
Message (RRR): Converged successfully.
Residual (RRR): [6.397726792783942e-11, 3.693401140481001e-11, -1.6626700016786344e-12]
FK at solution (RRR): [205.9484688 118.9044039 100.5872749]
Validation error norm (RRR): 7.386704679795208e-11

Method comparison on the same target:
- lm     converged=True iter=7 error=7.387e-11
- newton converged=True iter=6 error=1.642e-11
- ccd    converged=True iter=216 error=9.503e-09

Note: solve_position_ik handles position only (x, y, z), not full orientation IK.


## Solve inverse kinematics for a cartesian trajectory

In [31]:
from moro.inverse_kinematics import solve_position_trajectory
from moro.visualization import RobotVisualizer, VisualizationStyle

robot = mr.Robot(
    (l1, 0, 0, q1, "r"),
    (l2, 0, 0, q2, "r"),
)

# Numerical values for parameters
params = {l1: 200, l2: 200}

# Cartesian targets
targets = [[50*t,50*np.cos(3*t)+200,0] for t in np.linspace(-np.pi,np.pi)]

# Initial guess
q0 = [0.5, 0.5]

trajectory = solve_position_trajectory(
    robot,
    targets,
    q0=q0,
    parameters=params,
    method="ccd",
    tol=1e-8,
    max_iter=200,
)

print("Converged:", trajectory.converged)
print("Failed index:", trajectory.failed_index)
print("Message:", trajectory.message)

Converged: True
Failed index: None
Message: Trajectory solved successfully.


In [32]:
# Create a list of dicts to use with animate method
num_vals_list = []
for q in trajectory.qs:
    num_vals_list.append({
        q1: q[0],
        q2: q[1],
        l1: params[l1],
        l2: params[l2],
    })

viz = RobotVisualizer(robot)
viz.animate(num_vals_list, backend="threejs", style=VisualizationStyle(show_trajectory=True))